# `gold.fact_monthly_performance` — DDL

**Grain: one row per (ticker, month, manager). The only fact.**

The atomic grain the sources support. Every dimension joins it by a single key and none of
them needs a bridge to get here, so the model is a pure star.

The many-to-many between trusts and managers is resolved by **lowering the grain**: a trust
with six managers contributes six rows for the same month. That is what lets `dim_manager`
join directly, and it is why `allocation_factor` exists.

**This fact is not additive.** `AVG(total_return)` over it over-weights multi-manager
trusts. Take `DISTINCT` on (ticker, month) for a per-trust figure, or weight by
`allocation_factor` for anything that has to reconcile.

Declared here rather than inside the load, so the shape can be read without reading the
code that fills it. Catalog name has hyphens, so every reference needs backticks.

In [ ]:
CREATE TABLE IF NOT EXISTS `index-vs-trust-pipeline`.gold.fact_monthly_performance (
  monthly_key          STRING  COMMENT 'MD5(ticker|month_key|manager_key)',
  ticker_key           STRING  COMMENT 'The dim_ticker version live in this month, resolved at build time',
  month_key            INT     COMMENT 'Joins dim_date',
  manager_key          STRING  COMMENT 'Joins dim_manager. NotApplicable or NoInfo where there is no person',
  management_group_key STRING  COMMENT 'Joins dim_management_group',
  mandate_key          STRING  COMMENT 'Joins dim_mandate: region, asset class and style',
  ticker               STRING  COMMENT 'Readable, so the fact can be eyeballed without a join',
  close                DOUBLE  COMMENT 'Repaired where Bronze recorded an impossible price',
  dividend             DOUBLE,
  price_return         DOUBLE  COMMENT 'Null across a gap, never zero',
  total_return         DOUBLE  COMMENT 'Null in each series first month',
  allocation_factor    DOUBLE  COMMENT 'One over the manager count. THE WEIGHT THAT UNDOES THE FAN-OUT',
  manager_position     INT     COMMENT '1 is the first name listed. Source order, which is not seniority',
  return_basis         STRING  COMMENT 'Both sides always measured the same way'
)
COMMENT 'Monthly total return for every listed trust and the index, at the grain (ticker, month, manager)';

## Verification

Expected: the table exists, with the column count stated in
`specs/11_monthly_fact/monthly-fact.md` — **14**.

In [ ]:
SELECT COUNT(*) AS columns
FROM `index-vs-trust-pipeline`.information_schema.columns
WHERE table_schema = 'gold' AND table_name = 'fact_monthly_performance';